# RF-DETR Instance Segmentation (Nano) — Video Inference

Runs a Roboflow **RF-DETR Seg Preview (Nano)** `.pt` checkpoint over an uploaded `.mp4` and exports an annotated MP4 with instance masks + labels.

**Before running:** in Colab, set `Runtime → Change runtime type → GPU` (T4 is fine).

## Step 1: Install dependencies

In [ ]:
!pip install -q rfdetr supervision opencv-python-headless

## Step 2: Upload your weights (`.pt`) and the video (`.mp4`)

In [ ]:
from google.colab import files
import os

print('Upload your RF-DETR Seg Nano weights (.pt) AND your input video (.mp4):')
uploaded = files.upload()

weights_path = next(f for f in uploaded if f.lower().endswith('.pt'))
video_path   = next(f for f in uploaded if f.lower().endswith(('.mp4', '.mov', '.avi', '.mkv')))
output_path  = 'annotated_' + os.path.splitext(video_path)[0] + '.mp4'

print(f'Weights: {weights_path}')
print(f'Video:   {video_path}')
print(f'Output:  {output_path}')

## Step 3: Load the model and optimize for fast inference

`optimize_for_inference()` switches to fp16 + torch.compile, which roughly doubles FPS on a T4.

In [ ]:
import torch
from rfdetr import RFDETRSegPreview

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

# Nano variant of the seg-preview checkpoint
model = RFDETRSegPreview(pretrain_weights=weights_path, variant='nano')

# fp16 + compile for max throughput (no-op on CPU)
try:
    model.optimize_for_inference()
    print('Model optimized (fp16 + compile).')
except Exception as e:
    print(f'optimize_for_inference skipped: {e}')

# Class name lookup: model checkpoints trained on Roboflow carry COCO-style indices
class_names = getattr(model, 'class_names', None) or getattr(model.model, 'class_names', None)
print(f'Classes: {class_names}')

## Step 4: Run inference frame-by-frame and write the annotated video

In [ ]:
import cv2
import numpy as np
import supervision as sv
from tqdm import tqdm

CONF_THRESHOLD = 0.5

video_info = sv.VideoInfo.from_video_path(video_path)
print(f'Video: {video_info.width}x{video_info.height} @ {video_info.fps:.1f}fps, {video_info.total_frames} frames')

mask_annotator  = sv.MaskAnnotator(opacity=0.5)
box_annotator   = sv.BoxAnnotator(thickness=2)
label_annotator = sv.LabelAnnotator(text_scale=0.5, text_thickness=1)

def annotate(frame, detections):
    labels = [
        f"{class_names[c] if class_names and c < len(class_names) else c} {s:.2f}"
        for c, s in zip(detections.class_id, detections.confidence)
    ]
    out = mask_annotator.annotate(frame.copy(), detections)
    out = box_annotator.annotate(out, detections)
    out = label_annotator.annotate(out, detections, labels)
    return out

frames = sv.get_video_frames_generator(video_path)
with sv.VideoSink(output_path, video_info) as sink:
    for frame in tqdm(frames, total=video_info.total_frames, desc='Inferring'):
        detections = model.predict(frame, threshold=CONF_THRESHOLD)
        sink.write_frame(annotate(frame, detections))

print(f'Wrote {output_path}')

## Step 5: Preview and download

In [ ]:
from IPython.display import HTML
from base64 import b64encode

# Re-encode to H.264 so the Colab inline player can show it
preview_path = 'preview_' + output_path
!ffmpeg -y -loglevel error -i "{output_path}" -vcodec libx264 -pix_fmt yuv420p -crf 23 "{preview_path}"

mp4 = open(preview_path, 'rb').read()
HTML(f'<video width=720 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

In [ ]:
files.download(output_path)

## Speed tips
- **Downscale first** if your MP4 is 4K: add `frame = cv2.resize(frame, (1280, 720))` before `model.predict`.
- Raise `CONF_THRESHOLD` (e.g. 0.6) to skip low-confidence masks and shrink the annotation cost.
- On an A100/L4, expect ~2–3× the T4 throughput.
- First frame is slow — `torch.compile` warm-up. Subsequent frames hit steady-state FPS.